# Universal Turing Machine Simulator

A Turing machine is a tiny mathematical computer: it has a tape, a head, a current state, and a transition table.

Alan Turing proposed this model in 1936 to reason about what calculation itself means, before modern electronic computers existed. The idea still anchors computer science because it separates the essence of computation from any particular machine.

The wild part is that this simple model can compute anything that is computable. In this notebook, we will build a Python simulator that runs a Turing-machine program as data.

<details>
<summary>Big idea</summary>

Modern software looks huge, but at the foundation it is still state plus memory plus rules. A Turing machine makes that idea visible one symbol at a time.

</details>

## 1. Mental Model

A Turing machine has four moving parts:

- **Tape**: memory, split into cells that hold symbols like `1`, `+`, or `_`.
- **Head**: the pointer that reads one tape cell at a time.
- **State**: the machine's current mode, like `find_separator`.
- **Transition table**: the program. It says: when in this state and reading this symbol, write a symbol, move, and change state.

<details>
<summary>Blank-symbol convention</summary>

The tape is conceptually infinite, so most cells are empty. We represent empty cells with `_` instead of storing an endless list.

</details>

## 2. Build the Objects

We will model the machine directly:

- `Tape` stores only the non-blank cells in a dictionary.
- `Head` remembers the current tape position.
- `Transition` is one rule in the program.
- `TuringProgram` stores the transition table.
- `TuringMachine` executes the program and records snapshots.

<details>
<summary>Implementation hint</summary>

A dictionary is better than a list for the tape because the head can move left into negative positions or far right without preallocating space.

</details>

**Example state.** Create `Symbol`, `State`, `Direction`, the concrete values used in the next run.


In [ ]:
from dataclasses import dataclass

from typing import Literal

Symbol = str

State = str

Direction = Literal["L", "R", "N"]


**State model.** Define `Tape`, the data structure the lesson will operate on.


In [ ]:
@dataclass
class Tape:
    cells: dict[int, Symbol]
    blank: Symbol = "_"

    @classmethod
    def from_symbols(cls, symbols: list[Symbol], blank: Symbol = "_") -> "Tape":
        cells = {position: symbol for position, symbol in enumerate(symbols) if symbol != blank}
        return cls(cells=cells, blank=blank)

    def read(self, position: int) -> Symbol:
        return self.cells.get(position, self.blank)

    def write(self, position: int, symbol: Symbol) -> None:
        if symbol == self.blank:
            self.cells.pop(position, None)
        else:
            self.cells[position] = symbol

    def render(self, head_position: int, radius: int = 8) -> str:
        start = head_position - radius
        end = head_position + radius
        positions = range(start, end + 1)
        index_line = " ".join(f"{position:>2}" for position in positions)
        tape_line = " ".join(f"{self.read(position):>2}" for position in positions)
        head_line = " ".join(" ^" if position == head_position else "  " for position in positions)
        return f"pos : {index_line}\ntape: {tape_line}\nhead: {head_line}"

    def compact(self) -> str:
        if not self.cells:
            return self.blank
        start = min(self.cells)
        end = max(self.cells)
        return "".join(self.read(position) for position in range(start, end + 1)).strip(self.blank) or self.blank


**Object model.** Define `Head`, `Transition`, the named objects used by the next examples.


In [ ]:
@dataclass
class Head:
    position: int = 0

    def move(self, direction: Direction) -> None:
        if direction == "L":
            self.position -= 1
        elif direction == "R":
            self.position += 1

@dataclass(frozen=True)
class Transition:
    write: Symbol
    move: Direction
    next_state: State


**Algorithm engine.** Define `MachineSnapshot`, the class that runs the main simulation or algorithm.


In [ ]:
@dataclass(frozen=True)
class MachineSnapshot:
    step: int
    state: State
    head_position: int
    read_symbol: Symbol
    transition: Transition | None
    tape_view: str

    def summary(self) -> str:
        if self.transition is None:
            return f"step {self.step}: state={self.state}, read={self.read_symbol!r}, no rule found"
        return (
            f"step {self.step}: state={self.state}, read={self.read_symbol!r} -> "
            f"write={self.transition.write!r}, move={self.transition.move}, "
            f"next={self.transition.next_state}"
        )


**Object model.** Define `TuringProgram`, the named objects used by the next examples.


In [ ]:
@dataclass
class TuringProgram:
    transitions: dict[tuple[State, Symbol], Transition]
    start_state: State
    halt_states: tuple[State, ...] = ("HALT",)

    def transition_for(self, state: State, symbol: Symbol) -> Transition | None:
        return self.transitions.get((state, symbol))

    def is_halting(self, state: State) -> bool:
        return state in self.halt_states or state.startswith("HALT_")


**Algorithm engine.** Define `TuringMachine`, the class that runs the main simulation or algorithm.


In [ ]:
class TuringMachine:
    def __init__(self, program: TuringProgram, tape: Tape, head: Head | None = None):
        self.program = program
        self.tape = tape
        self.head = head or Head()
        self.state = program.start_state
        self.step_count = 0
        self.snapshots: list[MachineSnapshot] = []

    def halted(self) -> bool:
        return self.program.is_halting(self.state)

    def step(self) -> MachineSnapshot:
        symbol = self.tape.read(self.head.position)
        transition = self.program.transition_for(self.state, symbol)

        if transition is None:
            self.step_count += 1
            snapshot = MachineSnapshot(
                step=self.step_count,
                state=self.state,
                head_position=self.head.position,
                read_symbol=symbol,
                transition=None,
                tape_view=self.tape.render(self.head.position),
            )
            self.state = "HALT_NO_RULE"
            self.snapshots.append(snapshot)
            return snapshot

        current_state = self.state
        current_position = self.head.position
        self.tape.write(current_position, transition.write)
        self.head.move(transition.move)
        self.state = transition.next_state
        self.step_count += 1

        snapshot = MachineSnapshot(
            step=self.step_count,
            state=current_state,
            head_position=current_position,
            read_symbol=symbol,
            transition=transition,
            tape_view=self.tape.render(self.head.position),
        )
        self.snapshots.append(snapshot)
        return snapshot

    def run(self, max_steps: int = 100) -> list[MachineSnapshot]:
        while not self.halted() and self.step_count < max_steps:
            self.step()
        return self.snapshots


## 3. Program: Unary Addition

We will add numbers written in unary.

`111+11` means `3 + 2`.

The machine will turn it into `11111`, which means `5`.

The trick:

1. Move right until the `+` separator.
2. Replace `+` with `1`, temporarily joining the groups.
3. Move to the end.
4. Erase one extra `1` so the total is correct.

<details>
<summary>Why erase one symbol?</summary>

Changing `+` into `1` adds one extra mark. Erasing the last `1` balances that out, leaving exactly left plus right marks.

</details>

In [2]:
addition_program = TuringProgram(
    start_state="find_separator",
    transitions={
        ("find_separator", "1"): Transition(write="1", move="R", next_state="find_separator"),
        ("find_separator", "+"): Transition(write="1", move="R", next_state="scan_to_end"),
        ("scan_to_end", "1"): Transition(write="1", move="R", next_state="scan_to_end"),
        ("scan_to_end", "_"): Transition(write="_", move="L", next_state="erase_extra"),
        ("erase_extra", "1"): Transition(write="_", move="N", next_state="HALT"),
    },
)

print("Unary addition transition table")
print("state            read   write  move   next")
print("-" * 49)
for (state, symbol), transition in addition_program.transitions.items():
    print(
        f"{state:<16} {symbol:<6} {transition.write:<6} "
        f"{transition.move:<6} {transition.next_state}"
    )

Unary addition transition table
state            read   write  move   next
-------------------------------------------------
find_separator   1      1      R      find_separator
find_separator   +      1      R      scan_to_end
scan_to_end      1      1      R      scan_to_end
scan_to_end      _      _      L      erase_extra
erase_extra      1      _      N      HALT


## 4. Run the Machine

Now we place `111+11` on the tape and let the same simulator interpret the transition table.

This is the universal-machine idea in miniature: the simulator does not know addition. It only knows how to follow rules.

In [3]:
input_tape = Tape.from_symbols(list("111+11"))
machine = TuringMachine(program=addition_program, tape=input_tape)

print("Initial tape")
print(machine.tape.render(machine.head.position, radius=7))

snapshots = machine.run(max_steps=30)

print("\nFinal tape")
print(machine.tape.render(machine.head.position, radius=7))
print(f"\nHalted in state: {machine.state}")
print(f"Steps executed: {machine.step_count}")
print(f"Compact output: {machine.tape.compact()}")

Initial tape
pos : -7 -6 -5 -4 -3 -2 -1  0  1  2  3  4  5  6  7
tape:  _  _  _  _  _  _  _  1  1  1  +  1  1  _  _
head:                       ^                     

Final tape
pos : -2 -1  0  1  2  3  4  5  6  7  8  9 10 11 12
tape:  _  _  1  1  1  1  1  _  _  _  _  _  _  _  _
head:                       ^                     

Halted in state: HALT
Steps executed: 8
Compact output: 11111


## 5. Replay the Head Movement

A Turing machine is easiest to understand when you watch the head crawl across the tape.

Each replay line shows the current state, symbol read, symbol written, movement direction, and next state.

In [4]:
class MachineReplay:
    def __init__(self, snapshots: list[MachineSnapshot]):
        self.snapshots = snapshots

    def show(self) -> None:
        for snapshot in self.snapshots:
            print(snapshot.summary())
            print(snapshot.tape_view)
            print()


MachineReplay(snapshots).show()

step 1: state=find_separator, read='1' -> write='1', move=R, next=find_separator
pos : -7 -6 -5 -4 -3 -2 -1  0  1  2  3  4  5  6  7  8  9
tape:  _  _  _  _  _  _  _  1  1  1  +  1  1  _  _  _  _
head:                          ^                        

step 2: state=find_separator, read='1' -> write='1', move=R, next=find_separator
pos : -6 -5 -4 -3 -2 -1  0  1  2  3  4  5  6  7  8  9 10
tape:  _  _  _  _  _  _  1  1  1  +  1  1  _  _  _  _  _
head:                          ^                        

step 3: state=find_separator, read='1' -> write='1', move=R, next=find_separator
pos : -5 -4 -3 -2 -1  0  1  2  3  4  5  6  7  8  9 10 11
tape:  _  _  _  _  _  1  1  1  +  1  1  _  _  _  _  _  _
head:                          ^                        

step 4: state=find_separator, read='+' -> write='1', move=R, next=scan_to_end
pos : -4 -3 -2 -1  0  1  2  3  4  5  6  7  8  9 10 11 12
tape:  _  _  _  _  1  1  1  1  1  1  _  _  _  _  _  _  _
head:                          ^                 

## 6. Decode the Answer

Unary is simple: count the `1` symbols left on the tape.

<details>
<summary>Why use unary?</summary>

Unary is inefficient, but it makes the machine behavior obvious. More realistic encodings are possible, but the transition table gets bigger quickly.

</details>

In [5]:
def unary_value(tape: Tape) -> int:
    return sum(1 for symbol in tape.cells.values() if symbol == "1")


answer = unary_value(machine.tape)
print(f"{machine.tape.compact()} contains {answer} one-symbols.")
print(f"So 111+11 computed as {answer}.")

11111 contains 5 one-symbols.
So 111+11 computed as 5.


## 7. Playground: Add Other Unary Numbers

Now wrap the setup in a function. The transition table stays the same; only the starting tape changes.

**Builder helper.** Define `make_unary_addition_tape`, `run_unary_addition`, which prepares reusable examples or traces.


In [ ]:
def make_unary_addition_tape(left: int, right: int) -> Tape:
    if left < 0 or right < 0:
        raise ValueError("Unary addition only accepts non-negative integers.")
    return Tape.from_symbols(list("1" * left + "+" + "1" * right))

def run_unary_addition(left: int, right: int, max_steps: int = 100) -> dict[str, int | str]:
    trial_machine = TuringMachine(
        program=addition_program,
        tape=make_unary_addition_tape(left, right),
    )
    trial_machine.run(max_steps=max_steps)
    return {
        "left": left,
        "right": right,
        "steps": trial_machine.step_count,
        "output_tape": trial_machine.tape.compact(),
        "answer": unary_value(trial_machine.tape),
        "halt_state": trial_machine.state,
    }


**Run the experiment.** Advance the algorithm and collect the state changes that make the behavior visible.


In [ ]:
for left, right in [(1, 1), (2, 4), (5, 3), (0, 4)]:
    result = run_unary_addition(left, right)
    print(
        f"{result['left']} + {result['right']} -> "
        f"{result['output_tape']:<10} answer={result['answer']} "
        f"steps={result['steps']} halt={result['halt_state']}"
    )


## 8. Why This Is Universal

A universal Turing machine is a machine that can read the description of another machine and simulate it.

Our Python simulator plays that role here:

- The `TuringMachine` object is the interpreter.
- The `addition_program` dictionary is the program.
- The tape is the input and output memory.

Swap the transition table, and the same simulator runs a different machine.

<details>
<summary>Course connection</summary>

This is the same pattern you have seen across the course: define state, define rules, run steps, save snapshots, and inspect the process.

</details>

## 9. What You Should Remember

- A Turing machine has memory, a head, states, and rules.
- The transition table is the program.
- The machine does not understand addition; it only follows tiny local instructions.
- Complicated computation can emerge from very small mechanical steps.
- Universal computation means one machine can simulate many other machines when their rules are encoded as data.

<details>
<summary>Next experiment</summary>

Try writing a new transition table that flips every `1` to `0` until it reaches a blank. You can reuse the same `Tape`, `Head`, and `TuringMachine` classes.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Model computation as state, tape, and transition rules.

**Interactive animation target.** Animate the tape head, current state, read symbol, write symbol, and movement direction.

**Correctness handle.** At each step, the machine configuration is fully described by state, tape, and head position.

**Complexity handle.** Discuss steps until halting; some machines have no finite bound because they do not halt.

**Failure mode to test.** A tiny transition-table mistake can change halting behavior completely.

**Studio task.** Create a two-state machine, trace it for 20 steps, and state whether you expect it to halt.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
